# 02 — Classify: Sensitivity + risk scoring (0..100)

## Google Drive Setup

In [1]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Check if drive is already mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Drive is already mounted.")

print("Done!")

# Filepath Search
#search_term = "your_filename.ext"  # Change this
search_term = "synthetic_assets.csv"
!find /content/drive/MyDrive -maxdepth 15 -type f -iname "synthetic_assets.csv" -print


Mounted at /content/drive
Drive is already mounted.
Done!
/content/drive/MyDrive/data/synthetic_assets.csv


In [2]:
!pip -q install pandas numpy matplotlib  # safe to re-run

In [3]:
import pandas as pd
assets = pd.read_csv('/content/drive/MyDrive/data/synthetic_assets.csv')

WEIGHTS = {
  'sens': {'public':5,'internal':20,'confidential':45,'restricted':65},
  'internet': 25,
  'pii': 10,
  'phi': 15,
  'enc_none': 15,
  'enc_partial': 7,
}

def risk(row):
    s = WEIGHTS['sens'][row['sensitivity']]
    if bool(row['internet_exposed']): s += WEIGHTS['internet']
    if bool(row['has_pii']): s += WEIGHTS['pii']
    if bool(row['has_phi']): s += WEIGHTS['phi']
    if row['encryption'] == 'none': s += WEIGHTS['enc_none']
    elif row['encryption'] in ('at_rest','in_transit'): s += WEIGHTS['enc_partial']
    return min(100, int(s))

assets['risk'] = assets.apply(risk, axis=1)
assets.sort_values('risk', ascending=False).head(15)

,asset_id,asset_type,owner_team,region,sensitivity,has_pii,has_phi,encryption,internet_exposed,identity_plane,created_date,retention_days,risk
11,a011,saas,ml,ap-south,restricted,True,True,at_rest,True,api_gateway,2024-05-10,90,100
53,a053,warehouse,payments,us-west,restricted,True,True,none,False,api_gateway,2025-08-06,90,100
32,a032,vector_db,payments,us-west,restricted,True,False,both,True,entra_okta,2025-01-20,90,100
23,a023,db,ml,us-west,restricted,True,True,none,False,api_gateway,2024-09-18,90,100
46,a046,vector_db,ml,us-east,restricted,True,True,at_rest,False,entra_okta,2024-05-17,90,97
69,a069,object_store,salesops,us-west,restricted,True,True,in_transit,False,entra_okta,2024-08-15,90,97
24,a024,saas,ml,us-east,restricted,True,True,at_rest,False,api_gateway,2025-05-25,90,97
67,a067,warehouse,payments,eu-west,restricted,False,True,none,False,service_mesh,2024-10-08,90,95
47,a047,object_store,salesops,us-west,restricted,True,True,both,False,iam_workload,2025-02-16,90,90
65,a065,object_store,ops,us-west,restricted,True,False,none,False,iam_workload,2024-05-07,90,90


In [4]:
assets.groupby('sensitivity')['risk'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
sensitivity,,,,,,,,
confidential,25.0,60.9,7.7,45.0,55.0,62.0,62.0,80.0
internal,29.0,28.0,5.1,20.0,27.0,27.0,35.0,35.0
public,5.0,16.8,4.4,12.0,12.0,20.0,20.0,20.0
restricted,21.0,89.2,9.6,65.0,82.0,90.0,97.0,100.0
